In [1]:
from pathlib import Path
import json
import shutil

LAST_NAME = "Mejos"
STUDENT_ID = "TUPM-26-3605"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_summary.json"
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_report.txt"

target_year = 2015 + SEED_DIGIT

total_records = 0
incident_count = 0
invalid = 0
total_loss = 0.0
type_counts = {}
rows = []
losses = []

with open(dataset_path, "r", encoding="utf-8") as file:
    header = file.readline()

    with open(filtered_path, "w", encoding="utf-8") as output:
        output.write(header)

        for line in file:
            total_records += 1
            fields = line.strip().split(",")

            if len(fields) != 10:
                invalid += 1
            elif fields[1] == f"{target_year}":
                try:
                    loss = float(fields[4])
                    incident_count += 1
                    total_loss += loss
                    rows.append(fields)
                    losses.append(loss)
                    output.write(line.strip() + "\n")

                    attack = fields[2]
                    if attack in type_counts:
                        type_counts[attack] += 1
                    else:
                        type_counts[attack] = 1
                except ValueError:
                    invalid += 1

average_loss = total_loss / incident_count if incident_count else 0

# Five costliest incidents (highest loss first; ties keep the original order)
top_five = []
for _ in range(5):
    best_index = -1
    best_loss = -1.0

    for i in range(len(losses)):
        if losses[i] > best_loss:
            best_loss = losses[i]
            best_index = i

    if best_index != -1:
        top_five.append({
            "country": rows[best_index][0],
            "attack_type": rows[best_index][2],
            "target_industry": rows[best_index][3],
            "financial_loss": best_loss
        })
        losses[best_index] = -1.0

# JSON summary (includes the five costliest incidents) and backup
summary = {
    "target_year": target_year,
    "incidents_in_target_year": incident_count,
    "count_per_attack_type": type_counts,
    "total_financial_loss": round(total_loss, 2),
    "average_financial_loss": round(average_loss, 2),
    "five_costliest_incidents": top_five
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)

# Text report
with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-----------------------------\n")
    report.write(f"Target Year: {target_year}\n")
    report.write(f"Incidents: {incident_count}\n")
    report.write(f"Count per Attack Type: {type_counts}\n")
    report.write(f"Total Financial Loss (USD millions): {round(total_loss, 2)}\n")
    report.write(f"Average Financial Loss (USD millions): {round(average_loss, 2)}\n")
    report.write("Five Costliest Incidents:\n")
    for incident in top_five:
        report.write(f"  {incident['country']} | {incident['attack_type']} | {incident['financial_loss']}\n")
    report.write(f"Filtered CSV: {filtered_path.name}\n")
    report.write(f"JSON Summary: {json_path.name}\n")
    report.write(f"Backup File: {backup_path.name}\n")

print("Target year:", target_year)
print("Total records in dataset:", total_records)
print("Incidents in target year:", incident_count)
print("Invalid records:", invalid)
print("Count per attack type:", type_counts)
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))
print("Five costliest incidents:")
for incident in top_five:
    print(" ", incident)
print("Filtered CSV file:", filtered_path)
print("JSON summary file:", json_path)
print("Backup verification:", backup_path, "| exists:", backup_path.exists())
print("Report file:", report_path)

print("\n=== FINAL OUTPUT ===")
print(f"{incident_count} incidents occurred in {target_year}, with a total loss of ${round(total_loss, 2)} million.")
print(f"Report saved to {report_path.name}")

Target year: 2020
Total records in dataset: 3000
Incidents in target year: 315
Invalid records: 0
Count per attack type: {'Malware': 37, 'Phishing': 61, 'Man-in-the-Middle': 47, 'DDoS': 56, 'SQL Injection': 55, 'Ransomware': 59}
Total financial loss (USD millions): 15767.95
Average financial loss (USD millions): 50.06
Five costliest incidents:
  {'country': 'Brazil', 'attack_type': 'Ransomware', 'target_industry': 'Banking', 'financial_loss': 99.9}
  {'country': 'Japan', 'attack_type': 'DDoS', 'target_industry': 'Education', 'financial_loss': 99.83}
  {'country': 'USA', 'attack_type': 'DDoS', 'target_industry': 'Banking', 'financial_loss': 99.81}
  {'country': 'India', 'attack_type': 'SQL Injection', 'target_industry': 'Government', 'financial_loss': 99.72}
  {'country': 'USA', 'attack_type': 'Phishing', 'target_industry': 'Retail', 'financial_loss': 99.64}
Filtered CSV file: output\Mejos_TUPM-26-3605_incidents_filtered.csv
JSON summary file: output\Mejos_TUPM-26-3605_incidents_summary